# ROS2 mission runbook

Run these commands in separate TheConstruct shell windows. This notebook is documentation; it has not been executed in a Rosject.

## Task 2: TheConstruct setup

Target platform: **ROS2 Humble with Gazebo Classic**, not ROS1 Noetic or modern Gazebo/ros-gz. Choose a Rosject with the ROS2 TurtleBot3 Gazebo packages installed. Use the proposed name in ROSJECT_NAME.txt and verify it is available. This name is not an existing deployed project.

Upload/extract this package into /home/user/rsp_project and open Task2_Runbook.ipynb inside the Rosject. The commands below assume that exact directory. If the extracted folder differs, adjust that one path.

### Terminal 1: dependency preflight and build

```bash
source /opt/ros/humble/setup.bash
ros2 pkg prefix gazebo_ros
ros2 pkg prefix turtlebot3_gazebo
cd /home/user/rsp_project/ros2_ws
colcon build --symlink-install --packages-select warehouse_inspector
source install/setup.bash
ros2 launch warehouse_inspector simulation.launch.py
```

Keep Terminal 1 open. Open Gazebo's graphical window and confirm a Burger robot at the home marker, two inspection markers, shelves, barriers and landmarks. The launch file uses the installed TurtleBot3 Burger SDF. If preflight fails, use a matching Rosject; these instructions do not install or repair the hosted environment automatically. Do not launch two copies of the simulation.

### Terminal 2: verify interfaces, then launch the controller

```bash
source /opt/ros/humble/setup.bash
source /home/user/rsp_project/ros2_ws/install/setup.bash
ros2 topic list -t
ros2 topic info /scan --verbose
ros2 topic info /odom --verbose
ros2 topic info /cmd_vel --verbose
ros2 launch warehouse_inspector mission.launch.py
```

Required message types are sensor_msgs/msg/LaserScan, nav_msgs/msg/Odometry and geometry_msgs/msg/Twist respectively. Names can be changed in config/mission.yaml. A TwistStamped interface requires an adapter; do not use this node unchanged in that environment. Only this controller should command velocity during a mission. Sensor subscribers use the sensor-data QoS profile.

### Terminal 3: keyboard interface

```bash
source /opt/ros/humble/setup.bash
source /home/user/rsp_project/ros2_ws/install/setup.bash
ros2 run warehouse_inspector keyboard
```

Type `start` and press Enter. Other commands are `pause`, `resume`, `cancel`, `reset`, and `quit`. The keyboard is a line-based interface so it works in browser terminals. `quit` sends cancel. Start requires fresh, valid scan and odometry. A FAULT needs `reset` followed by `start`; it never resumes automatically.

### Terminal 4: monitor and capture evidence

```bash
source /opt/ros/humble/setup.bash
source /home/user/rsp_project/ros2_ws/install/setup.bash
ros2 topic echo /inspection/status
```

CSV logs are written to /home/user/inspection_evidence. Save screenshots of IDLE, navigation, inspection, avoidance and COMPLETE with the relevant terminal visible. Export the actual CSV and copy it into evidence/theconstruct; do not replace it with the offline test CSV.

### Required execution checks

- Normal mission: two inspection points, then home; expect COMPLETE. Inspect the trajectory and CSV.
- Pause/resume: pause while moving, verify zero velocity command, then resume.
- Cancel: cancel while moving, verify IDLE and zero velocity command.
- Obstacle: use Gazebo's object insertion tool to place a box ahead. Verify the stop and AVOID transition. Record recovery or FAULT honestly. Capture the box position.
- Stale sensors: pause Gazebo physics via its UI while the mission is active. The wall-clock watchdog should stop commands and enter FAULT after the sensor timeout. Unpause, reset and rerun. Availability of pause services varies by environment.
- Unreachable goal: select a blocked goal and record avoidance or mission timeout. Do not claim a global planner will find an alternative route.

